# Experiment: Two-Phase Ricci-Based Architecture Screening

**Dataset**: Fashion-MNIST (Sneakers vs Sandals)

**Hypothesis**: A Ricci coefficient that remains shallow (close to 0) after a short
probe run signals that the architecture is *geometrically immature* it has not yet
begun curvature-driven flow and therefore needs a larger training budget rather
than being discarded.

## Protocol

| Phase | What happens |
|-------|-------------|
| **Phase 1 Probe** | Train ALL 45 architectures for exactly `PHASE1_EPOCHS` epochs (no early stopping). At the end, compute each architecture's robust avg Ricci score. |
| **Phase 2 Extended** | Continue training ALL 45 architectures from their Phase 1 weights up to `TOTAL_EPOCHS` epochs. No Ricci computation in this phase pure training. |

**Key question**: Do the geometrically immature architectures eventually reach
comparable accuracy when given more training budget?



In [ ]:
import os, time, json, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, TensorDataset
from scipy.sparse import csr_matrix, lil_matrix
from scipy.sparse import triu as sp_triu
from scipy.sparse.csgraph import dijkstra
from sklearn.neighbors import kneighbors_graph
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'Device: {device}')

In [ ]:
PHASE1_EPOCHS = 10       # Short probe: all 45 architectures run for this many epochs
TOTAL_EPOCHS  = 200      # Total training budget (Phase 2 continues from Phase 1 up to this)


CKPT_EVERY = 5

# ── Selection criterion ──
# Architectures whose robust avg Ricci after Phase 1 is ABOVE this threshold
# are considered geometrically immature and selected for Phase 2.
# "Above" = closer to 0 = not yet flowing.
RICCI_SELECTION_THRESHOLD = -0.6

# ── Standard early stopping (passive observer in Phase 2)
PATIENCE = 5

# ── kNN
K_FRAC = 0.05            # k = 5% of test-set size (paper Appendix A.5.1)

# ── Curvature type
CURVATURE_TYPE = 'Augmented-Forman-Ricci'

# ── Training ──
LR = 0.001
BATCH_SIZE = 128

# ── Architecture grid
FLAT_DEPTHS      = [4, 5, 6, 7, 8, 9, 10, 11, 12]
FLAT_WIDTHS      = [16, 32, 64, 128]
BOTTLENECK_DEPTHS = [4, 5, 6, 7, 8, 9, 10, 11, 12]
BOTTLENECK_WIDTH  = 128

ARCHITECTURES = []
for d in FLAT_DEPTHS:
    for w in FLAT_WIDTHS:
        ARCHITECTURES.append((f'flat_{d}_{w}', 'flat', d, w))
for d in BOTTLENECK_DEPTHS:
    ARCHITECTURES.append((f'bottleneck_{d}_{BOTTLENECK_WIDTH}', 'bottleneck', d, BOTTLENECK_WIDTH))

print(f'Total architectures: {len(ARCHITECTURES)}')
print(f'Flat: {len(FLAT_DEPTHS)*len(FLAT_WIDTHS)}, Bottleneck: {len(BOTTLENECK_DEPTHS)}')

# ── Output paths
DRIVE_BASE   = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'results')
OUTPUT_DIR   = os.path.join(DRIVE_BASE, 'two_phase_experiment')
PHASE1_DIR   = os.path.join(OUTPUT_DIR, 'phase1')
PHASE2_DIR   = os.path.join(OUTPUT_DIR, 'phase2')
PHASE1_CKPT  = os.path.join(OUTPUT_DIR, 'phase1_checkpoint.json')
PHASE2_CKPT  = os.path.join(OUTPUT_DIR, 'phase2_checkpoint.json')

for d in [OUTPUT_DIR, PHASE1_DIR, PHASE2_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Output root: {OUTPUT_DIR}')

In [ ]:
# ── Load Fashion-MNIST (Sneakers=7 vs Sandals=5)
transform  = transforms.Compose([transforms.ToTensor()])
train_data = torchvision.datasets.FashionMNIST(root='./data', train=True,  download=True, transform=transform)
test_data  = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

def filter_classes(dataset, class_a=5, class_b=7):
    mask = (dataset.targets == class_a) | (dataset.targets == class_b)
    X    = dataset.data[mask].float() / 255.0
    y    = dataset.targets[mask].float()
    y    = (y == class_b).float()   # Sneakers=1, Sandals=0
    return X.to(device), y.to(device)

X_train, y_train = filter_classes(train_data)
X_test,  y_test  = filter_classes(test_data)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Batches per epoch: {len(train_loader)}')
print(f'Class balance — Train: {y_train.mean():.3f}, Test: {y_test.mean():.3f}')

# ── kNN k = 5% of test set (paper Appendix A.5.1) ──
K = max(1, int(round(K_FRAC * X_test.shape[0])))
print(f'kNN k = {K}  (= {K_FRAC*100:.0f}% of {X_test.shape[0]} test samples)')


In [ ]:
def get_bottleneck_widths(depth, max_width):
    """Generate symmetric bottleneck widths.
    E.g. depth=5, max_width=128 -> [128, 64, 32, 64, 128]
    """
    mid = depth // 2
    widths = []
    for i in range(depth):
        dist_from_mid = abs(i - mid)
        if depth % 2 == 0:
            dist_from_mid  = abs(i - mid + 0.5) + 0.5
            dist_from_edge = min(i, depth - 1 - i)
        else:
            dist_from_edge = min(i, depth - 1 - i)
        reduction = 2 ** (mid - dist_from_edge) if dist_from_edge < mid else 1
        w = max(max_width // reduction, 8)
        widths.append(w)
    return widths


class DNN(nn.Module):
    def __init__(self, input_dim=784, layer_widths=None, hidden_units=None, depth=None):
        super().__init__()
        if layer_widths is None:
            layer_widths = [hidden_units] * depth
        self.flatten = nn.Flatten()

        dims = [input_dim] + layer_widths + [1]

        layers = []
        for i in range(len(dims) - 1):
            linear = nn.Linear(dims[i], dims[i + 1])
            nn.init.kaiming_normal_(linear.weight, nonlinearity='relu')
            nn.init.zeros_(linear.bias)
            layers.append(linear)
        self.layers     = nn.ModuleList(layers)
        self.batchnorms = nn.ModuleList([
            nn.BatchNorm1d(dims[i + 1]) for i in range(len(dims) - 2)
        ])

    def forward(self, x):
        x = self.flatten(x)
        for i, layer in enumerate(self.layers[:-1]):
            x = layer(x)
            x = self.batchnorms[i](x)
            x = torch.relu(x)
        return self.layers[-1](x)

    def features(self, x):
        
        x = self.flatten(x)
        feats = [x.detach().cpu().numpy()]           # index 0: flattened input
        for i, layer in enumerate(self.layers[:-1]):
            x = layer(x)
            x = self.batchnorms[i](x)
            x = torch.relu(x)
            feats.append(x.detach().cpu().numpy())   # indices 1..L: hidden layers
        return feats


def build_model(arch_type, depth, width):
    if arch_type == 'bottleneck':
        widths = get_bottleneck_widths(depth, width)
        model  = DNN(input_dim=784, layer_widths=widths)
    else:
        model  = DNN(input_dim=784, hidden_units=width, depth=depth)
    return model.to(device)


# Smoke test
_m = build_model('flat', 5, 64)
print(f'Flat 5/64: {[(l.in_features, l.out_features) for l in _m.layers]}')
del _m
print('\u2705 DNN defined')


In [ ]:
def accuracy_fn(y_true, y_pred):
    return (torch.eq(y_true, y_pred).sum().item() / len(y_pred)) * 100

def train_one_epoch(model, train_loader, optimizer, loss_fn):
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in train_loader:
        y_logits = model(X_batch).squeeze()
        loss = loss_fn(y_logits, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(X_batch)
    return total_loss / len(train_loader.dataset)

def evaluate(model, X, y):
    model.eval()
    with torch.inference_mode():
        y_logits = model(X).squeeze()
        y_preds  = torch.round(torch.sigmoid(y_logits))
        acc  = accuracy_fn(y, y_preds) / 100.0
        loss = nn.BCEWithLogitsLoss()(y_logits, y).item()
    return acc, loss

print('\u2705 Training utilities defined')


In [ ]:
def compute_forman_ricci(A):
    A       = csr_matrix(A)
    degrees = A @ np.ones(A.shape[0])
    Ric     = lil_matrix(A.shape, dtype=np.int32)
    rows, cols = sp_triu(A, k=1).nonzero()
    for i, j in zip(rows, cols):
        Ric[i, j] = 4 - degrees[i] - degrees[j]
        Ric[j, i] = Ric[i, j]
    return Ric.tocsr()

def compute_augmented_forman_ricci(A):
    A       = csr_matrix(A)
    degrees = A @ np.ones(A.shape[0])
    A2      = A @ A
    Ric     = lil_matrix(A.shape, dtype=np.int32)
    rows, cols = sp_triu(A, k=1).nonzero()
    for i, j in zip(rows, cols):
        Ric[i, j] = 4 - degrees[i] - degrees[j] + 3 * A2[i, j]
        Ric[j, i] = Ric[i, j]
    return Ric.tocsr()

def compute_approx_ollivier_ricci(A):
    A       = csr_matrix(A)
    degrees = A @ np.ones(A.shape[0])
    A2      = A @ A
    Ric     = lil_matrix(A.shape, dtype=np.float32)
    rows, cols = sp_triu(A, k=1).nonzero()
    for i, j in zip(rows, cols):
        t  = A2[i, j]; di, dj = degrees[i], degrees[j]
        Ric[i, j] = (0.5 * (t / max(di, dj))
                     - 0.5 * (max(0, 1 - 1/di - 1/dj - t/min(di, dj))
                               + max(0, 1 - 1/di - 1/dj - t/max(di, dj))
                               - t / max(di, dj)))
        Ric[j, i] = Ric[i, j]
    return Ric.tocsr()

def compute_curvature(A, curv):
    if   curv == 'Forman-Ricci':            return compute_forman_ricci(A)
    elif curv == 'Augmented-Forman-Ricci':  return compute_augmented_forman_ricci(A)
    elif curv == 'Approx-Ollivier-Ricci':   return compute_approx_ollivier_ricci(A)
    else: raise ValueError(f'Unknown curvature: {curv}')

def compute_ricci_from_features(features, k, curv='Augmented-Forman-Ricci'):
    """
    Compute layer Ricci coefficients (paper Section 3.2).
    Returns array of shape (len(features)-1,).
    """
    depth = len(features)
    kNN_graphs = []
    for feat in features:
        g = kneighbors_graph(feat, k, mode='connectivity', include_self=False)
        kNN_graphs.append(g.maximum(g.T))
    apsps      = [dijkstra(csgraph=g, directed=False, unweighted=True,
                           return_predecessors=False) for g in kNN_graphs]
    curvatures = [compute_curvature(kNN_graphs[i], curv) for i in range(depth - 1)]

    layer_ricci = np.empty(depth - 1)
    for i in range(depth - 1):
        sc, eta = [], []
        for x in range(len(features[0])):
            S1 = kNN_graphs[i][x].indices
            ec, connected = 0, True
            for y in S1:
                if apsps[i + 1][x, y] == np.inf:
                    connected = False
                else:
                    ec += apsps[i + 1][x, y] - apsps[i][x, y]
            if connected:
                sc.append(np.divide(curvatures[i][x].sum(),
                                    kNN_graphs[i][x].count_nonzero()))
                eta.append(ec / len(S1))
        layer_ricci[i] = pearsonr(sc, eta)[0] if len(sc) >= 2 else np.nan
    return layer_ricci

print('\u2705 Curvature functions defined')


In [ ]:
def robust_avg_ricci_iqr(layer_ricci, iqr_factor=1.5):
    valid_mask = np.isfinite(layer_ricci)
    valid = layer_ricci[valid_mask]
    if len(valid) < 3:
        return float(np.nanmean(layer_ricci)), []
    q1, q3 = np.percentile(valid, [25, 75])
    upper_bound = q3 + iqr_factor * (q3 - q1)
    excluded, kept = [], []
    for i, val in enumerate(layer_ricci):
        if not np.isfinite(val): continue
        (excluded if val > upper_bound else kept).append((i, val) if val > upper_bound else val)
    # redo cleanly
    excluded_indices, kept_values = [], []
    for i, val in enumerate(layer_ricci):
        if not np.isfinite(val): continue
        if val > upper_bound: excluded_indices.append(i)
        else: kept_values.append(val)
    if not kept_values:
        return float(np.nanmean(layer_ricci)), []
    return float(np.mean(kept_values)), excluded_indices

def robust_avg_ricci_threshold(layer_ricci, threshold=-0.6):
    """Average over layers whose Ricci is below threshold.
    threshold : float, default=-0.6
        Layers with Ricci >= threshold are considered geometrically degenerate.
    """
    excluded_indices, kept_values = [], []
    for i, val in enumerate(layer_ricci):
        if not np.isfinite(val): continue
        if val >= threshold: excluded_indices.append(i)
        else: kept_values.append(val)
    if not kept_values:
        return float(np.nanmean(layer_ricci)), []
    return float(np.mean(kept_values)), excluded_indices

print('\u2705 Robust averaging functions defined (available for post-hoc analysis)')
print('  Note: Phase 1 scoring and Phase 2 tracking use plain nanmean — no filtering.')
print('        Filtering layers would hide the geometric immaturity we are trying to detect.')


In [ ]:
class StandardEarlyStopping:
    """Passive observer: tracks when patience-based ES on test accuracy would stop.
    Patience counter activates only after acc changes from its initial value."""
    def __init__(self, patience=5):
        self.patience    = patience
        self.initial_acc = None
        self.activated   = False
        self.best_acc    = -1.0
        self.best_epoch  = None
        self.wait        = 0
        self.stop_epoch  = None

    def step(self, epoch, val_acc):
        if self.stop_epoch is not None:
            return
        if self.initial_acc is None:
            self.initial_acc = float(val_acc)
            self.best_acc    = float(val_acc)
            self.best_epoch  = epoch
            return
        if not self.activated:
            if val_acc != self.initial_acc:
                self.activated  = True
                self.best_acc   = float(val_acc)
                self.best_epoch = epoch
                self.wait       = 0
            return
        if val_acc > self.best_acc:
            self.best_acc   = float(val_acc)
            self.best_epoch = epoch
            self.wait       = 0
        else:
            self.wait += 1
        if self.wait >= self.patience:
            self.stop_epoch = epoch

print('\u2705 StandardEarlyStopping defined')


In [ ]:
def load_json(path, default):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return default

def save_json(path, obj):
    with open(path, 'w') as f:
        json.dump(obj, f, indent=2)

def save_activations(epoch_dir, hidden_acts):
    """Save per-hidden-layer activations as individual .npy files."""
    for i, act in enumerate(hidden_acts):
        np.save(os.path.join(epoch_dir, f'hidden_{i}.npy'), act)

print('\u2705 Checkpoint / IO utilities defined')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 1 — PROBE RUN  (PHASE1_EPOCHS epochs, all 45 architectures)
# ═══════════════════════════════════════════════════════════════════════════════
p1_ckpt      = load_json(PHASE1_CKPT, {'completed': []})
p1_completed = set(p1_ckpt['completed'])
p1_summary_path = os.path.join(OUTPUT_DIR, 'phase1_summary.json')
p1_summary      = load_json(p1_summary_path, [])
t0 = time.time()
for arch_idx, (name, arch_type, depth, width) in enumerate(ARCHITECTURES):
    if name in p1_completed:
        print(f'[{arch_idx+1:2d}/{len(ARCHITECTURES)}] {name} — skipped (checkpoint)')
        continue
    print(f'\n[{arch_idx+1:2d}/{len(ARCHITECTURES)}] {name}  (type={arch_type}, depth={depth}, width={width})')
    arch_dir = os.path.join(PHASE1_DIR, name)
    os.makedirs(arch_dir, exist_ok=True)
    model    = build_model(arch_type, depth, width)
    loss_fn  = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    epoch_history = []
    per_epoch_ricci = []

    # ── Check for mid-training checkpoint (Colab resume)
    start_epoch = 0
    arch_ckpt_path = os.path.join(arch_dir, 'training_checkpoint.pth')
    ckpt_meta_path = os.path.join(arch_dir, 'training_checkpoint.json')
    if os.path.exists(arch_ckpt_path) and os.path.exists(ckpt_meta_path):
        ckpt = torch.load(arch_ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        meta = load_json(ckpt_meta_path, {})
        start_epoch     = meta['epoch'] + 1
        epoch_history   = meta['epoch_history']
        per_epoch_ricci = meta['per_epoch_ricci']
        print(f'  Resuming from epoch {start_epoch}')

    for epoch in range(start_epoch, PHASE1_EPOCHS):
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn)
        test_acc, test_loss = evaluate(model, X_test, y_test)
        train_acc, _        = evaluate(model, X_train, y_train)
        model.eval()
        with torch.inference_mode():
            feats = model.features(X_test)
        hidden_acts_ep = feats[1:]
        layer_ricci = compute_ricci_from_features(hidden_acts_ep, K, curv=CURVATURE_TYPE)
        avg_ricci_epoch = float(np.nanmean(layer_ricci))
        per_epoch_ricci.append(avg_ricci_epoch)
        if (epoch + 1) % 5 == 0 or epoch == PHASE1_EPOCHS - 1:
            print(f'  ep {epoch:3d}: train={train_acc:.4f}  test={test_acc:.4f}  ricci={avg_ricci_epoch:+.4f}')
        epoch_history.append({
            'epoch':      epoch,
            'train_acc':  float(train_acc),
            'test_acc':   float(test_acc),
            'train_loss': float(train_loss),
            'test_loss':  float(test_loss),
            'avg_ricci':  avg_ricci_epoch,
        })
        # ── Save mid-training checkpoint for Colab
        if (epoch + 1) % CKPT_EVERY == 0:
            torch.save({
                'model_state':     model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
            }, arch_ckpt_path)
            save_json(ckpt_meta_path, {
                'epoch':           epoch,
                'epoch_history':   epoch_history,
                'per_epoch_ricci': per_epoch_ricci,
            })

    # ── Extract final Ricci and activations
    model.eval()
    with torch.inference_mode():
        feats = model.features(X_test)
    hidden_acts = feats[1:]
    layer_ricci_final = compute_ricci_from_features(hidden_acts, K, curv=CURVATURE_TYPE)
    avg_ricci      = float(np.nanmean(per_epoch_ricci))
    final_test_acc = epoch_history[-1]['test_acc']
    print(f'  → avg_ricci (over {PHASE1_EPOCHS} epochs)={avg_ricci:+.4f}  final_test_acc={final_test_acc:.4f}')

    # Save model weights AND optimizer state (for Phase 2 resumption)
    torch.save(model.state_dict(), os.path.join(arch_dir, 'model_weights_phase1.pth'))
    torch.save(optimizer.state_dict(), os.path.join(arch_dir, 'optimizer_phase1.pth'))

    # Remove mid-training checkpoint (no longer needed)
    for p in [arch_ckpt_path, ckpt_meta_path]:
        if os.path.exists(p):
            os.remove(p)

    act_dir = os.path.join(arch_dir, 'activations_phase1')
    os.makedirs(act_dir, exist_ok=True)
    save_activations(act_dir, hidden_acts)
    np.save(os.path.join(arch_dir, 'layer_ricci_phase1.npy'), layer_ricci_final)
    np.save(os.path.join(arch_dir, 'per_epoch_ricci_phase1.npy'), np.array(per_epoch_ricci))
    save_json(os.path.join(arch_dir, 'epoch_history_phase1.json'), epoch_history)
    entry = {
        'name':                  name,
        'type':                  arch_type,
        'depth':                 depth,
        'width':                 width,
        'avg_ricci_phase1':      avg_ricci,
        'per_epoch_ricci':       per_epoch_ricci,
        'layer_ricci_phase1':    layer_ricci_final.tolist(),
        'final_test_acc_phase1': float(final_test_acc),
        'selected_for_phase2':   True,
    }
    p1_summary.append(entry)
    save_json(p1_summary_path, p1_summary)
    p1_completed.add(name)
    save_json(PHASE1_CKPT, {'completed': list(p1_completed)})
    del model, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
n_selected = sum(1 for r in p1_summary if r['selected_for_phase2'])
print(f'\n Phase 1 complete in {time.time()-t0:.0f}s')
print(f'   {n_selected}/{len(p1_summary)} architectures selected for Phase 2')

In [ ]:
# ── Phase 1 results: all 45 architectures sorted by avg_ricci
p1_summary = load_json(os.path.join(OUTPUT_DIR, 'phase1_summary.json'), [])
if not p1_summary:
    print('Run Phase 1 first.')
else:
    sorted_p1 = sorted(p1_summary, key=lambda r: r['avg_ricci_phase1'])
    print(f'Phase 1 results — {len(sorted_p1)} architectures sorted by avg_ricci')
    print(f'(avg_ricci = mean of {PHASE1_EPOCHS} per-epoch Ricci values)')
    print()
    print(f'{"Rank":<5} {"Architecture":<26} {"avg_ricci":>10} {"final_acc":>10}')
    print('-' * 55)
    for rank, r in enumerate(sorted_p1, 1):
        print(f'{rank:<5} {r["name"]:<26} {r["avg_ricci_phase1"]:>+10.4f} {r["final_test_acc_phase1"]:>10.4f}')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 2 — EXTENDED RUN  (continue from Phase 1 up to TOTAL_EPOCHS epochs)
# Models resume from their Phase 1 weights — no retraining from scratch.
# No Ricci computation in this phase — pure training.
# ═══════════════════════════════════════════════════════════════════════════════
p1_summary  = load_json(os.path.join(OUTPUT_DIR, 'phase1_summary.json'), [])
selected    = [r for r in p1_summary if r['selected_for_phase2']]
if not selected:
    print('No architectures selected or run Phase 1 first.')
else:
    p2_ckpt      = load_json(PHASE2_CKPT, {'completed': []})
    p2_completed = set(p2_ckpt['completed'])
    p2_summary_path = os.path.join(OUTPUT_DIR, 'phase2_summary.json')
    p2_summary      = load_json(p2_summary_path, [])
    t0 = time.time()
    for run_idx, p1_entry in enumerate(selected):
        name      = p1_entry['name']
        arch_type = p1_entry['type']
        depth     = p1_entry['depth']
        width     = p1_entry['width']
        if name in p2_completed:
            print(f'[{run_idx+1:2d}/{len(selected)}] {name} — skipped (checkpoint)')
            continue
        print(f'\n[{run_idx+1:2d}/{len(selected)}] {name}  (type={arch_type}, depth={depth}, width={width})')
        print(f'  Phase 1 avg_ricci={p1_entry["avg_ricci_phase1"]:+.4f}')
        arch_dir = os.path.join(PHASE2_DIR, name)
        os.makedirs(arch_dir, exist_ok=True)

        # ── Build model & load Phase 1 weights (continue training)
        model     = build_model(arch_type, depth, width)
        loss_fn   = nn.BCEWithLogitsLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)

        p1_weights_path = os.path.join(PHASE1_DIR, name, 'model_weights_phase1.pth')
        p1_optim_path   = os.path.join(PHASE1_DIR, name, 'optimizer_phase1.pth')
        model.load_state_dict(torch.load(p1_weights_path, map_location=device, weights_only=False))
        optimizer.load_state_dict(torch.load(p1_optim_path, map_location=device, weights_only=False))
        print(f'  Loaded Phase 1 weights — continuing from epoch {PHASE1_EPOCHS}')

        # Load Phase 1 epoch history (build a continuous training curve)
        p1_history = load_json(os.path.join(PHASE1_DIR, name, 'epoch_history_phase1.json'), [])
        epoch_history = list(p1_history)

        start_epoch = PHASE1_EPOCHS

        # ── Check for mid-training checkpoint (Colab resume)
        arch_ckpt_path = os.path.join(arch_dir, 'training_checkpoint.pth')
        ckpt_meta_path = os.path.join(arch_dir, 'training_checkpoint.json')
        if os.path.exists(arch_ckpt_path) and os.path.exists(ckpt_meta_path):
            ckpt = torch.load(arch_ckpt_path, map_location=device, weights_only=False)
            model.load_state_dict(ckpt['model_state'])
            optimizer.load_state_dict(ckpt['optimizer_state'])
            meta = load_json(ckpt_meta_path, {})
            start_epoch   = meta['epoch'] + 1
            epoch_history = meta['epoch_history']
            print(f'  Resuming from epoch {start_epoch}')

        for epoch in range(start_epoch, TOTAL_EPOCHS):
            train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn)
            test_acc,  test_loss  = evaluate(model, X_test,  y_test)
            train_acc, _ = evaluate(model, X_train, y_train)
            epoch_history.append({
                'epoch':        epoch,
                'train_acc':    float(train_acc),
                'test_acc':     float(test_acc),
                'train_loss':   float(train_loss),
                'test_loss':    float(test_loss),
            })
            if (epoch + 1) % 20 == 0 or epoch == TOTAL_EPOCHS - 1:
                print(f'  ep {epoch:4d}: train={train_acc:.4f}  test={test_acc:.4f}')
            # ── Save mid-training checkpoint for Colab
            if (epoch + 1) % CKPT_EVERY == 0:
                torch.save({
                    'model_state':     model.state_dict(),
                    'optimizer_state': optimizer.state_dict(),
                }, arch_ckpt_path)
                save_json(ckpt_meta_path, {
                    'epoch':         epoch,
                    'epoch_history': epoch_history,
                })

        # ── Save final results ──
        torch.save(model.state_dict(), os.path.join(arch_dir, 'model_weights_final.pth'))
        save_json(os.path.join(arch_dir, 'epoch_history_full.json'), epoch_history)

        # Remove mid-training checkpoint
        for p in [arch_ckpt_path, ckpt_meta_path]:
            if os.path.exists(p):
                os.remove(p)

        final_acc = float(epoch_history[-1]['test_acc'])
        summary_entry = {
            'name':                   name,
            'type':                   arch_type,
            'depth':                  depth,
            'width':                  width,
            'avg_ricci_phase1':       p1_entry['avg_ricci_phase1'],
            'acc_phase1':             p1_entry['final_test_acc_phase1'],
            'final_acc_phase2':       final_acc,
            'total_epochs':           len(epoch_history),
        }
        p2_summary.append(summary_entry)
        save_json(p2_summary_path, p2_summary)
        p2_completed.add(name)
        save_json(PHASE2_CKPT, {'completed': list(p2_completed)})
        del model, optimizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print(f'\n✅ Phase 2 complete in {time.time()-t0:.0f}s')
    print(f'   Results saved to {p2_summary_path}')

In [ ]:
# ── Phase 2 results: summary table sorted by final_acc
p2_summary = load_json(os.path.join(OUTPUT_DIR, 'phase2_summary.json'), [])
if not p2_summary:
    print('Run Phase 2 first.')
else:
    sorted_p2 = sorted(p2_summary, key=lambda r: r['final_acc_phase2'], reverse=True)
    hdr = (f'{"Rank":<5} {"Architecture":<26} {"P1 Ricci":>9} {"Final Acc":>10}')
    print(hdr)
    print('-' * len(hdr))
    for rank, r in enumerate(sorted_p2, 1):
        print(f'{rank:<5} {r["name"]:<26} '
              f'{r["avg_ricci_phase1"]:>+9.4f} '
              f'{r["final_acc_phase2"]:>10.4f}')


In [ ]:
# Cross-phase comparison: Ricci vs Final Accuracy Ranking
p2_summary = load_json(os.path.join(OUTPUT_DIR, 'phase2_summary.json'), [])
if not p2_summary:
    print('Run Phase 2 first.')
else:
    # Rank by Phase 1 Ricci (ascending = smaller is better/more mature geometrically)
    sorted_by_ricci = sorted(p2_summary, key=lambda r: r['avg_ricci_phase1'])
    ricci_rank = {r['name']: i+1 for i, r in enumerate(sorted_by_ricci)}
    
    # Rank by Phase 2 Final Acc (descending = larger is better)
    sorted_by_acc = sorted(p2_summary, key=lambda r: r['final_acc_phase2'], reverse=True)
    acc_rank = {r['name']: i+1 for i, r in enumerate(sorted_by_acc)}
    
    print(f'{"Architecture":<26} {"Ricci Rank":>10} {"Acc Rank":>10} {"Diff":>6}')
    print('-' * 55)
    # Loop over architectures by their Ricci rank
    for r in sorted_by_ricci:
        name = r['name']
        rr = ricci_rank[name]
        ar = acc_rank[name]
        print(f'{name:<26} {rr:>10} {ar:>10} {rr - ar:>6}')
    
    # Optionally compute Spearman rank correlation
    import scipy.stats as stats
    r_ranks = [ricci_rank[r['name']] for r in p2_summary]
    a_ranks = [acc_rank[r['name']] for r in p2_summary]
    correlation, pval = stats.spearmanr(r_ranks, a_ranks)
    print(f'\nSpearman Rank Correlation: {correlation:.4f} (p-value: {pval:.4f})')


In [ ]:
# Save combined results to CSV
p1_summary = load_json(os.path.join(OUTPUT_DIR, 'phase1_summary.json'), [])
p2_summary = load_json(os.path.join(OUTPUT_DIR, 'phase2_summary.json'), [])

p2_lookup = {r['name']: r for r in p2_summary}

rows = []
for p1 in p1_summary:
    name = p1['name']
    row = {
        'name':                  name,
        'type':                  p1['type'],
        'depth':                 p1['depth'],
        'width':                 p1['width'],
        'avg_ricci_phase1':      p1['avg_ricci_phase1'],
        'final_test_acc_phase1': p1['final_test_acc_phase1'],
        'selected_for_phase2':   p1['selected_for_phase2'],
    }
    # Add per-layer Ricci values as separate columns
    for i, lr in enumerate(p1.get('layer_ricci_phase1', [])):
        row[f'layer_ricci_{i}'] = lr

    p2 = p2_lookup.get(name)
    if p2:
        row['final_acc_phase2'] = p2['final_acc_phase2']
        row['total_epochs']     = p2['total_epochs']
    rows.append(row)

df = pd.DataFrame(rows)
csv_path = os.path.join(OUTPUT_DIR, 'two_phase_results.csv')
df.to_csv(csv_path, index=False)
print(f' CSV saved to {csv_path}')
print(f'   {len(df)} rows, {len(df.columns)} columns')
df